# Fase 6 — Deploy

## 6.1 Política de pedido (conta aberta)

Cenário da Fase 4: histórico até 03/10/2019, promoção planejada dos 28 dias seguintes, estoque de segurança dos erros do SARIMAX nas 12 origens do backtest e 3.000 unidades no CD (o estoque do CD não está na base; `estoque_total` é estoque de loja).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from src.dados import carregar_demanda
from src.pedido import sugerir_pedido
from src.politica import custos_unitarios, nivel_servico_otimo
from src.preparacao import H, remover_descontinuados, separar_treino_teste, serie_produto

SEED = 42
np.random.seed(SEED)
RAIZ = Path.cwd().parent
PRODUTO = "P0131"
MODELO = "SARIMAX + promoção"
ESTOQUE_CD = 3000

serie_p0131 = serie_produto(remover_descontinuados(carregar_demanda()), PRODUTO)
historico, teste = separar_treino_teste(serie_p0131)
futuro = teste[["pct_lojas_promo"]]

backtest = pd.read_csv(RAIZ / "dados" / "processados" / "backtest_P0131.csv")
erros_sarimax = backtest.loc[backtest["modelo"] == MODELO].sort_values("origem_n")["erro_unidades"].values
nivel_servico = nivel_servico_otimo(*custos_unitarios(serie_p0131["preco_medio"].mean()))

print(f"Histórico: {historico.index.min().date()} a {historico.index.max().date()} | horizonte: {futuro.index.min().date()} a {futuro.index.max().date()}")
print(f"Dias de promoção > 20% no horizonte: {(futuro['pct_lojas_promo'] > 0.2).sum()} | nível de serviço: {nivel_servico:.2%}")

Histórico: 2017-01-02 a 2019-10-03 | horizonte: 2019-10-04 a 2019-10-31
Dias de promoção > 20% no horizonte: 0 | nível de serviço: 93.75%


In [2]:
pedido = sugerir_pedido(historico, futuro, erros_sarimax, ESTOQUE_CD, nivel_servico)

print(f"SUGESTÃO DE PEDIDO — {PRODUTO} | decisão em {historico.index.max():%d/%m/%Y} | cobre {futuro.index.min():%d/%m} a {futuro.index.max():%d/%m/%Y}")
print("-" * 78)
print(f"   Demanda prevista (28 dias, SARIMAX + promoção)   {pedido['demanda_prevista']:>10,.2f} un")
print(f" + Estoque de segurança (quantil {nivel_servico:.2%} dos erros) {pedido['estoque_seguranca']:>10,.2f} un")
print(f" = Necessidade bruta                                 {pedido['necessidade_bruta']:>10,.2f} un")
print(f" − Estoque disponível no CD                          {pedido['estoque_cd']:>10,.2f} un")
print(f" = PEDIDO AO FORNECEDOR                              {pedido['pedido']:>10,.2f} un  (≈ {np.ceil(pedido['pedido']):,.0f})")
print("-" * 78)

vendido = teste["unidades"].sum()
estoque_final = pedido["estoque_cd"] + pedido["pedido"] - vendido
print(f"\nCOMPARAÇÃO COM O REALIZADO")
print(f"   Vendido de fato nos 28 dias                       {vendido:>10,.2f} un")
print(f"   Demanda prevista − vendido                        {pedido['demanda_prevista'] - vendido:>+10,.2f} un ({(pedido['demanda_prevista'] - vendido) / vendido:+.2%})")
print(f"   Disponível (CD + pedido) − vendido                {estoque_final:>+10,.2f} un  -> {'sobra' if estoque_final >= 0 else 'falta'} no fim do período")
print(f"   Sobra final / estoque de segurança                {estoque_final / pedido['estoque_seguranca']:>10.1%}")

SUGESTÃO DE PEDIDO — P0131 | decisão em 03/10/2019 | cobre 04/10 a 31/10/2019
------------------------------------------------------------------------------
   Demanda prevista (28 dias, SARIMAX + promoção)     8,238.12 un
 + Estoque de segurança (quantil 93.75% dos erros)     786.11 un
 = Necessidade bruta                                   9,024.23 un
 − Estoque disponível no CD                            3,000.00 un
 = PEDIDO AO FORNECEDOR                                6,024.23 un  (≈ 6,025)
------------------------------------------------------------------------------

COMPARAÇÃO COM O REALIZADO
   Vendido de fato nos 28 dias                         8,158.00 un
   Demanda prevista − vendido                            +80.12 un (+0.98%)
   Disponível (CD + pedido) − vendido                   +866.23 un  -> sobra no fim do período
   Sobra final / estoque de segurança                    110.2%


In [3]:
semanas = pd.DataFrame({"previsto": pedido["previsao_diaria"], "vendido": teste["unidades"].values}, index=teste.index)
semanas = semanas.groupby(np.arange(H) // 7).agg(inicio=("previsto", lambda s: s.index.min()), previsto=("previsto", "sum"), vendido=("vendido", "sum"))
semanas["inicio"] = teste.index[::7].strftime("%d/%m")
semanas["diferenca"] = semanas["previsto"] - semanas["vendido"]
print("Previsto × vendido por semana do horizonte:")
print(semanas.round(1).to_string(index=False))

es_sem_janela = np.quantile(erros_sarimax[:-1], nivel_servico)
print(f"\nSensibilidade: estoque de segurança sem a origem 12 (que é a própria janela de teste): {es_sem_janela:.2f} un "
      f"-> pedido {max(pedido['demanda_prevista'] + es_sem_janela - ESTOQUE_CD, 0):,.2f} un")

Previsto × vendido por semana do horizonte:
inicio  previsto  vendido  diferenca
 04/10    2035.4   2028.0        7.4
 11/10    2065.6   2087.0      -21.4
 18/10    2067.6   1977.0       90.6
 25/10    2069.5   2066.0        3.5

Sensibilidade: estoque de segurança sem a origem 12 (que é a própria janela de teste): 831.53 un -> pedido 6,069.65 un


## 6.2 Produção

O fluxo deste notebook virou o script `sugerir_pedido.py` (raiz do repositório), que usa `src/pipeline.py`:

```
python sugerir_pedido.py --produto P0131 --estoque-cd 3000 --calendario exemplos/calendario_promo_exemplo.csv
```

A cada execução ele recalcula o backtest com as 12 origens que terminam até a data da decisão, retreina o SARIMAX com todo o histórico, prevê os 28 dias seguintes com o calendário promocional informado e salva a conta aberta em `saidas/`.

Diferença em relação à célula 6.1: lá o estoque de segurança (786 un) usou as origens da Fase 5, que incluem a própria janela de teste. Em produção só entram janelas encerradas; simulando a decisão em 03/10/2019 as origens vão de 04/04 a 05/09, pegam a promoção de abril/2019 e o estoque de segurança sobe para 1.603 un.